# Orders ML Practices

A practical roadmap for diagnostic, predictive, prescriptive, generative/agentic AI, and big-data ML using `Orders.csv`.

> Local sections run against the available Orders data. Azure, streaming, Spark, and Delta Lake sections are implementation templates and require corresponding services or files.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

orders = pd.read_csv(Path('Orders.csv'))
for column in ['Order Date', 'Ship Date']:
    if column in orders:
        orders[column] = pd.to_datetime(orders[column], dayfirst=True, errors='coerce')
print(f'Loaded {len(orders):,} rows and {orders.shape[1]} columns')
display(orders.head())

## 1. Diagnostic Analytics

### Correlation and variance analysis

In [ ]:
numeric = orders.select_dtypes(include=np.number)
variance = numeric.var().sort_values(ascending=False).to_frame('variance')
display(variance)
plt.figure(figsize=(10, 7))
sns.heatmap(numeric.corr(), annot=True, fmt='.2f', cmap='vlag', center=0)
plt.title('Numeric Correlation Matrix')
plt.show()

### Root-cause analysis: negative profit and weak margins

This identifies dimensions associated with loss-making rows. Association is diagnostic evidence, not proof of causation.

In [ ]:
orders['Profit Margin'] = orders['Profit'] / orders['Sales'].replace(0, np.nan)
loss_rows = orders[orders['Profit'] < 0]
print(f'Loss-making rows: {len(loss_rows):,} ({len(loss_rows) / len(orders):.1%})')
for dimension in ['Category', 'Sub-Category', 'Region', 'Segment', 'Ship Mode']:
    if dimension in orders:
        root_cause = orders.groupby(dimension).agg(
            rows=('Profit', 'size'),
            loss_rows=('Profit', lambda x: (x < 0).sum()),
            sales=('Sales', 'sum'),
            profit=('Profit', 'sum'),
            avg_discount=('Discount', 'mean') if 'Discount' in orders else ('Profit', 'mean')
        )
        root_cause['loss_rate'] = root_cause['loss_rows'] / root_cause['rows']
        root_cause['margin'] = root_cause['profit'] / root_cause['sales']
        print(f'### {dimension}')
        display(root_cause.sort_values('loss_rate', ascending=False).head(10))

## 2. Predictive Analytics

The following models are baselines for experimentation. Use time-aware validation before production use.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

model_data = orders.dropna(subset=['Sales', 'Profit']).copy()
feature_columns = [c for c in ['Quantity', 'Discount', 'Ship Mode', 'Segment', 'Region', 'Category', 'Sub-Category'] if c in model_data]
X = model_data[feature_columns]
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
preprocess = ColumnTransformer([('num', 'passthrough', numeric_features), ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])

# Regression: estimate Profit.
X_train, X_test, y_train, y_test = train_test_split(X, model_data['Profit'], test_size=.2, random_state=42)
regression = Pipeline([('preprocess', preprocess), ('model', RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1))])
regression.fit(X_train, y_train)
prediction = regression.predict(X_test)
print(f'Profit regression MAE: {mean_absolute_error(y_test, prediction):,.2f} | R2: {r2_score(y_test, prediction):.3f}')

In [ ]:
# Classification: predict whether an order is profitable.
classification_target = (model_data['Profit'] > 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, classification_target, test_size=.2, random_state=42, stratify=classification_target)
classifier = Pipeline([('preprocess', preprocess), ('model', RandomForestClassifier(n_estimators=150, random_state=42, class_weight='balanced', n_jobs=-1))])
classifier.fit(X_train, y_train)
class_prediction = classifier.predict(X_test)
print(f'Profitability classification accuracy: {accuracy_score(y_test, class_prediction):.3f}')
print(classification_report(y_test, class_prediction, target_names=['Loss', 'Profit']))

### Forecasting sales trends and demand planning

In [ ]:
monthly = orders.dropna(subset=['Order Date']).assign(Month=lambda d: d['Order Date'].dt.to_period('M').dt.to_timestamp()).groupby('Month').agg(Sales=('Sales', 'sum'), Quantity=('Quantity', 'sum')).reset_index()
monthly['Sales_3M_MA'] = monthly['Sales'].rolling(3, min_periods=1).mean()
monthly['Demand_3M_MA'] = monthly['Quantity'].rolling(3, min_periods=1).mean()
display(monthly.tail(12))
monthly.plot(x='Month', y=['Sales', 'Sales_3M_MA'], figsize=(14, 5), title='Sales Trend and 3-Month Forecast Baseline'); plt.show()
monthly.plot(x='Month', y=['Quantity', 'Demand_3M_MA'], figsize=(14, 5), title='Demand Planning Baseline'); plt.show()

### Customer churn prediction

A customer is labeled as churned when their last observed order is at least 180 days before the dataset's latest order. This is a baseline label; production churn needs a business-approved observation window.

In [ ]:
if {'Customer ID', 'Order Date'}.issubset(orders.columns):
    latest_date = orders['Order Date'].max()
    customer_features = orders.groupby('Customer ID').agg(
        last_order=('Order Date', 'max'), orders=('Order ID', 'nunique'), sales=('Sales', 'sum'), profit=('Profit', 'sum'), avg_discount=('Discount', 'mean')
    ).reset_index()
    customer_features['recency_days'] = (latest_date - customer_features['last_order']).dt.days
    customer_features['churned'] = (customer_features['recency_days'] >= 180).astype(int)
    display(customer_features.head())
    print(f'Baseline churn rate: {customer_features.churned.mean():.1%}')
else:
    print('Customer churn requires Customer ID and Order Date.')

## 3. Prescriptive Analytics

### KPI dashboard, optimization model, and actionable recommendations

In [ ]:
kpis = pd.Series({
        'Total Sales': orders['Sales'].sum(),
        'Total Profit': orders['Profit'].sum(),
        'Profit Margin %': orders['Profit'].sum() / orders['Sales'].sum() * 100,
        'Orders': orders['Order ID'].nunique(),
        'Customers': orders['Customer ID'].nunique(),
        'Average Order Value': orders.groupby('Order ID')['Sales'].sum().mean(),
        'Average Shipping Days': (orders['Ship Date'] - orders['Order Date']).dt.days.mean()
    })
display(kpis.to_frame('value'))

# Simple constrained pricing/discount scenario: maximize expected profit per row under a discount cap.
if 'Discount' in orders:
    scenario = orders.groupby('Discount').agg(Sales=('Sales', 'sum'), Profit=('Profit', 'sum'), Rows=('Profit', 'size')).reset_index()
    scenario['Profit Margin %'] = scenario['Profit'] / scenario['Sales'] * 100
    display(scenario.sort_values('Profit Margin %', ascending=False))

recommendations = []
if 'Region' in orders:
    weak_region = (orders.groupby('Region')['Profit'].sum() / orders.groupby('Region')['Sales'].sum()).idxmin()
    recommendations.append(f'Review pricing, discounting, and product mix in the weakest-margin region: {weak_region}.')
if 'Sub-Category' in orders:
    loss_categories = orders.groupby('Sub-Category')['Profit'].sum().loc[lambda x: x < 0].index.tolist()
    recommendations.append(f'Investigate loss-making sub-categories: {", ".join(loss_categories)}.')
recommendations.append('Use the forecast and demand moving average to set replenishment targets and safety stock.')
display(pd.DataFrame({'Actionable recommendation': recommendations}))

## 4. Generative AI and Agentic AI

Recommended architecture: Azure AI Foundry/Azure OpenAI for model access, Azure AI Search for semantic search and knowledge grounding, LangGraph or an equivalent orchestrator for stateful workflows, and function calling for governed tools. Keep secrets in environment variables or Key Vault, validate tool arguments, log traces, and require human approval for high-impact actions.

In [ ]:
# Template only - requires Azure OpenAI credentials and the openai package.
# from openai import AzureOpenAI
# client = AzureOpenAI(
#     api_key=os.environ['AZURE_OPENAI_API_KEY'],
#     api_version=os.environ['AZURE_OPENAI_API_VERSION'],
#     azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'])
# response = client.chat.completions.create(
#     model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
#     messages=[{'role': 'user', 'content': 'Summarize the latest sales and profit trend.'}])
# print(response.choices[0].message.content)

ai_capabilities = pd.DataFrame({
    'Capability': ['Azure AI Foundry / Azure OpenAI', 'LangGraph / prompt orchestration', 'RAG / semantic search', 'Function calling / tools', 'Conversational or multi-agent systems', 'Human-in-the-loop', 'MLflow', 'Enterprise governance'],
    'Notebook status': ['Template', 'Design pattern', 'Design pattern', 'Design pattern', 'Design pattern', 'Pending integration', 'Pending integration', 'Pending architecture']
})
display(ai_capabilities)

## 5. Big Data ML

Delta Lake validation and Spark/Hive integration require a Delta table or Spark runtime. Feature engineering below is expressed in pandas and can be translated to Spark DataFrame operations at scale.

In [ ]:
# Local feature engineering baseline.
features = orders.copy()
features['Shipping Days'] = (features['Ship Date'] - features['Order Date']).dt.days
features['Order Month'] = features['Order Date'].dt.month
features['Order Quarter'] = features['Order Date'].dt.quarter
features['Sales per Unit'] = features['Sales'] / features['Quantity'].replace(0, np.nan)
features['Discounted Loss Flag'] = ((features['Discount'] > 0) & (features['Profit'] < 0)).astype(int)
display(features[['Shipping Days', 'Order Month', 'Order Quarter', 'Sales per Unit', 'Discounted Loss Flag']].describe().T)

# Template for Delta Lake validation (requires deltalake or Spark):
# from deltalake import DeltaTable
# delta_table = DeltaTable('path/to/orders_delta')
# delta_table.schema().json()
# delta_table.to_pandas().head()

big_data_status = pd.DataFrame({
        'Capability': ['Delta Lake validation', 'Feature engineering at scale', 'PyTorch/TensorFlow evaluation', 'Spark and Hive', 'Large-scale training/evaluation', 'Batch pipeline integration', 'Streaming pipeline integration', 'ML workflow performance optimization'],
        'Notebook status': ['Template', 'Local baseline; Spark-ready design', 'Pending model-specific dataset', 'Pending Spark runtime', 'Pending cluster/runtime', 'Pending orchestration', 'Pending streaming source', 'Pending profiling and tracking']
    })
display(big_data_status)